# 🔍 ForensicAI — Suspect Visualization System
### Feature-Based Facial Image Generation Using Latent Diffusion Models

**Authors:** Arpita Waghmare, Sampada Bandre, Lavanya Tiwade  
**Guide:** Prof. Riddhi Doshi  
**Institution:** Symbiosis Institute of Technology, Nagpur  
**Published:** ICACS 2026

---

### 📌 What This Notebook Does
1. Installs all required libraries
2. Loads Stable Diffusion v1.5 from HuggingFace
3. Generates **Photorealistic Mugshot** from text description
4. Generates **Forensic Pencil Sketch** from same description
5. Registers faces into a **Criminal Database**
6. Matches generated face against database using **Facenet512 + VGG-Face ensemble**
7. Returns **MATCH / NOT IN DATABASE** with similarity score

### ⚡ Runtime Required
- Go to **Runtime > Change runtime type > T4 GPU**
- Free Colab T4 is sufficient

---

## STEP 1 — Install Dependencies

In [ ]:
# Install all required packages
!pip install diffusers transformers accelerate torch torchvision -q
!pip install deepface tf-keras opencv-python pillow -q
!pip install gradio -q

print("✅ All packages installed!")

## STEP 2 — Import Libraries

In [ ]:
import torch
import numpy as np
import os
import json
import shutil
from datetime import datetime
from PIL import Image
import cv2
import gradio as gr

from diffusers import StableDiffusionPipeline
from deepface import DeepFace

# Check GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✅ Using device: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  No GPU found. Please enable T4 GPU in Runtime settings.")

## STEP 3 — Load Stable Diffusion v1.5 Model
> This will download ~4GB model weights from HuggingFace. Takes ~3-5 minutes on first run.

In [ ]:
from diffusers import StableDiffusionPipeline
import torch

MODEL_ID = "runwayml/stable-diffusion-v1-5"

print("Loading Stable Diffusion v1.5...")
pipe = StableDiffusionPipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    safety_checker=None,
    requires_safety_checker=False
)
pipe = pipe.to(device)

# Memory optimization
pipe.enable_attention_slicing()

print("✅ Model loaded successfully!")
print(f"   Text Encoder : CLIP ViT-L/14")
print(f"   Denoiser     : U-Net")
print(f"   Decoder      : VAE (KL-regularized)")

## STEP 4 — Prompt Engineering Module

| Mode | CFG Scale | Timesteps T |
|------|-----------|-------------|
| Mugshot | 8.5 | 40 |
| Sketch | 7.5 | 35 |

In [ ]:
def build_prompts(description, mode="Mugshot"):
    """
    Build structured positive and negative prompts for forensic output.
    
    Args:
        description (str): Natural language suspect description
        mode (str): 'Mugshot' or 'Sketch'
    
    Returns:
        tuple: (positive_prompt, negative_prompt, cfg_scale, timesteps)
    """
    if mode == "Mugshot":
        positive = (
            f"{description}, "
            "photorealistic mugshot, law enforcement photography, "
            "front-facing gaze, neutral expression, white background, "
            "sharp focus, 8K DSLR quality, highly detailed face, "
            "police identification photo, studio lighting"
        )
        negative = (
            "cartoon, anime, blur, watermark, accessories, smile, "
            "side profile, low quality, deformed, extra limbs, "
            "hat, glasses, sunglasses, painting, illustration, "
            "distorted, bad anatomy, ugly, duplicate"
        )
        cfg = 8.5
        steps = 40

    elif mode == "Sketch":
        positive = (
            f"{description}, "
            "forensic pencil sketch, charcoal on aged paper, "
            "police composite drawing, graphite shading, "
            "hatching technique, monochrome, high detail, "
            "forensic art, black and white portrait sketch"
        )
        negative = (
            "photorealistic, color, digital art, anime, "
            "watermark, blur, painting, oil, cartoonish, "
            "low quality, distorted, bad anatomy"
        )
        cfg = 7.5
        steps = 35

    else:
        raise ValueError("Mode must be 'Mugshot' or 'Sketch'")

    return positive, negative, cfg, steps


def generate_image(description, mode="Mugshot", seed=42):
    """
    Generate forensic image from text description.

    Core Formula:
        CFG: e_hat = e_u + w * (e_c - e_u)
        VAE: x_hat = VAE_decode(z_0 / 0.18215)

    Latent space: 64x64x4 = 16,384 values (vs pixel 512x512x3 = 786,432 → 48x faster)
    """
    pos_prompt, neg_prompt, cfg, steps = build_prompts(description, mode)

    generator = torch.Generator(device=device).manual_seed(seed)

    with torch.autocast(device):
        result = pipe(
            prompt=pos_prompt,
            negative_prompt=neg_prompt,
            num_inference_steps=steps,
            guidance_scale=cfg,
            height=512,
            width=512,
            generator=generator
        )

    image = result.images[0]
    return image


print("✅ Prompt Engineering module ready!")

## STEP 5 — Criminal Database Manager

In [ ]:
class CriminalDatasetManager:
    """
    Manages persistent criminal database.
    Stores: face photos, metadata (name, case ID, crime type, date)
    """

    def __init__(self, base_dir="/content/criminal_database"):
        self.base_dir = base_dir
        self.photos_dir = os.path.join(base_dir, "photos")
        self.metadata_file = os.path.join(base_dir, "criminals_metadata.json")
        os.makedirs(self.photos_dir, exist_ok=True)
        self._load_metadata()
        print(f"✅ Database initialized at: {base_dir}")
        print(f"   Total records: {len(self.metadata)}")

    def _load_metadata(self):
        if os.path.exists(self.metadata_file):
            with open(self.metadata_file, "r") as f:
                self.metadata = json.load(f)
        else:
            self.metadata = []

    def _save_metadata(self):
        with open(self.metadata_file, "w") as f:
            json.dump(self.metadata, f, indent=2)

    def register(self, name, case_id, crime_type, photo_path):
        """
        Register a new criminal record.
        """
        # Copy photo to database
        ext = os.path.splitext(photo_path)[1]
        dest = os.path.join(self.photos_dir, f"{case_id}{ext}")
        shutil.copy(photo_path, dest)

        record = {
            "name": name,
            "case_id": case_id,
            "crime_type": crime_type,
            "photo_path": dest,
            "registered_on": datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        }
        self.metadata.append(record)
        self._save_metadata()
        print(f"✅ Registered: {name} | Case ID: {case_id}")
        return record

    def list_records(self):
        print(f"\n{'='*50}")
        print(f"CRIMINAL DATABASE — {len(self.metadata)} records")
        print(f"{'='*50}")
        for r in self.metadata:
            print(f"  {r['case_id']} | {r['name']} | {r['crime_type']}")
        print(f"{'='*50}\n")

    def delete_record(self, case_id):
        self.metadata = [r for r in self.metadata if r["case_id"] != case_id]
        self._save_metadata()
        print(f"✅ Deleted record: {case_id}")


# Initialize database
db = CriminalDatasetManager()
print("✅ Criminal Database Manager ready!")

## STEP 6 — Face Matching Module

**Formula:**
```
similarity = max(0, (1 - cosine_distance) × 100)
S_ensemble = (S_Facenet512 + S_VGGFace) / 2

Decision:
  S_ensemble >= 60%  →  MATCH
  S_ensemble <  60%  →  NOT IN DATABASE
```

In [ ]:
def compute_similarity(img1_path, img2_path, model_name):
    """
    Compute cosine similarity between two face images.
    similarity = max(0, (1 - cosine_distance) * 100)
    """
    try:
        result = DeepFace.verify(
            img1_path=img1_path,
            img2_path=img2_path,
            model_name=model_name,
            distance_metric="cosine",
            enforce_detection=False
        )
        cosine_distance = result["distance"]
        similarity = max(0, (1 - cosine_distance) * 100)
        return round(similarity, 2)
    except Exception as e:
        print(f"   Warning ({model_name}): {e}")
        return 0.0


def match_against_database(generated_image, db, threshold=60.0):
    """
    Match generated image against all criminal database records.
    Uses Facenet512 + VGG-Face ensemble.

    Args:
        generated_image (PIL.Image): Generated suspect image
        db (CriminalDatasetManager): Criminal database
        threshold (float): Cosine similarity threshold (default 60%)

    Returns:
        dict: Match result with name, case_id, score, decision
    """
    if not db.metadata:
        return {"decision": "DATABASE EMPTY", "message": "No records in database."}

    # Save generated image temporarily
    temp_path = "/content/temp_generated.png"
    generated_image.save(temp_path)

    best_match = None
    best_score = 0.0
    all_results = []

    print("\nScanning criminal database...")
    print(f"{'Case ID':<12} {'Name':<20} {'Facenet512':>12} {'VGG-Face':>10} {'Ensemble':>10}")
    print("-" * 68)

    for record in db.metadata:
        db_photo = record["photo_path"]

        # Ensemble: Facenet512 + VGG-Face
        score_fn = compute_similarity(temp_path, db_photo, "Facenet512")
        score_vgg = compute_similarity(temp_path, db_photo, "VGG-Face")
        score_ensemble = round((score_fn + score_vgg) / 2, 2)

        print(f"{record['case_id']:<12} {record['name']:<20} {score_fn:>11.1f}% {score_vgg:>9.1f}% {score_ensemble:>9.1f}%")

        all_results.append({
            "name": record["name"],
            "case_id": record["case_id"],
            "crime_type": record["crime_type"],
            "score_facenet": score_fn,
            "score_vgg": score_vgg,
            "score_ensemble": score_ensemble
        })

        if score_ensemble > best_score:
            best_score = score_ensemble
            best_match = record
            best_ensemble = score_ensemble

    print("-" * 68)
    print(f"Threshold: {threshold}%")
    print()

    # Decision
    if best_score >= threshold:
        decision = "MATCH"
        result = {
            "decision": decision,
            "name": best_match["name"],
            "case_id": best_match["case_id"],
            "crime_type": best_match["crime_type"],
            "similarity_score": best_score,
            "all_results": all_results
        }
        print(f"🟢 MATCH FOUND!")
        print(f"   Name      : {best_match['name']}")
        print(f"   Case ID   : {best_match['case_id']}")
        print(f"   Crime     : {best_match['crime_type']}")
        print(f"   Score     : {best_score:.1f}%")
    else:
        decision = "NOT IN DATABASE"
        result = {
            "decision": decision,
            "closest_match": best_match["name"] if best_match else "N/A",
            "closest_score": best_score,
            "all_results": all_results
        }
        print(f"🔴 NOT IN DATABASE")
        print(f"   Closest   : {best_match['name'] if best_match else 'N/A'}")
        print(f"   Score     : {best_score:.1f}% (below {threshold}% threshold)")

    return result


print("✅ Face Matching Module ready!")

## STEP 7 — Register Faces into Criminal Database
> Upload real face photos and register them as criminal records.

In [ ]:
from google.colab import files

print("Upload a face photo to register in the criminal database:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\nUploaded: {filename}")
    
    # Fill in details
    name       = input("Enter name: ")
    case_id    = input("Enter Case ID (e.g. TM-001): ")
    crime_type = input("Enter crime type: ")

    db.register(
        name=name,
        case_id=case_id,
        crime_type=crime_type,
        photo_path=filename
    )

db.list_records()

## STEP 8 — Generate Forensic Images + Database Scan
> Enter your suspect description below and run!

In [ ]:
from IPython.display import display, Image as IPImage
import matplotlib.pyplot as plt

# ── ENTER SUSPECT DESCRIPTION HERE ──
SUSPECT_DESCRIPTION = "Female, young adult 20-30, South Asian, black hair, \
brown eyes, high cheekbones, plump lips, broad eyebrows, sharp nose"

print(f"Description: {SUSPECT_DESCRIPTION}")
print("\nGenerating images...")

# Generate both modes
print("[1/2] Generating Mugshot (CFG=8.5, T=40)...")
mugshot = generate_image(SUSPECT_DESCRIPTION, mode="Mugshot", seed=42)

print("[2/2] Generating Sketch (CFG=7.5, T=35)...")
sketch = generate_image(SUSPECT_DESCRIPTION, mode="Sketch", seed=42)

# Display both
fig, axes = plt.subplots(1, 2, figsize=(12, 6))
axes[0].imshow(mugshot)
axes[0].set_title("Photorealistic Mugshot", fontsize=14, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(sketch)
axes[1].set_title("Forensic Pencil Sketch", fontsize=14, fontweight="bold")
axes[1].axis("off")

plt.suptitle("ForensicAI — Dual Mode Generation", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("/content/forensicai_output.png", dpi=150, bbox_inches="tight")
plt.show()

print("\n✅ Images generated!")

# Save mugshot for matching
mugshot.save("/content/generated_mugshot.png")

In [ ]:
# Run Database Scan on generated mugshot
print("Running criminal database scan on generated mugshot...\n")

match_result = match_against_database(mugshot, db, threshold=60.0)

print("\n" + "=" * 40)
print("FINAL DECISION:", match_result["decision"])
print("=" * 40)

## STEP 9 — Gradio Web Interface (Optional)
> Run this cell to launch a full interactive web UI with public URL

In [ ]:
def forensicai_pipeline(description, output_style, db_scan):
    """
    Full ForensicAI pipeline for Gradio interface.
    """
    if not description.strip():
        return None, None, "Please enter a suspect description."

    # Generate images
    mugshot_img = generate_image(description, mode="Mugshot")
    sketch_img  = generate_image(description, mode="Sketch")

    result_text = "Images generated successfully.\n"

    # Database scan
    if db_scan and db.metadata:
        result = match_against_database(mugshot_img, db)
        if result["decision"] == "MATCH":
            result_text += (
                f"\n🟢 MATCH FOUND\n"
                f"Name      : {result['name']}\n"
                f"Case ID   : {result['case_id']}\n"
                f"Crime     : {result['crime_type']}\n"
                f"Similarity: {result['similarity_score']:.1f}%"
            )
        else:
            result_text += (
                f"\n🔴 NOT IN DATABASE\n"
                f"Closest Match : {result['closest_match']}\n"
                f"Score         : {result['closest_score']:.1f}%\n"
                f"(Below 60% threshold)"
            )
    elif db_scan and not db.metadata:
        result_text += "\n⚠️  Database is empty. Please register faces first."

    return mugshot_img, sketch_img, result_text


# Build Gradio Interface
with gr.Blocks(title="ForensicAI") as demo:
    gr.Markdown("# 🔍 ForensicAI — Suspect Visualization System")
    gr.Markdown("Enter a natural language description of the suspect to generate forensic images and scan the criminal database.")

    with gr.Row():
        with gr.Column():
            description = gr.Textbox(
                label="Suspect Description",
                placeholder="e.g. Male, 30-40 years, medium build, black hair, brown eyes, beard, scar on left cheek",
                lines=4
            )
            output_style = gr.Radio(
                choices=["Mugshot", "Sketch", "Both"],
                value="Both",
                label="Output Mode"
            )
            db_scan = gr.Checkbox(
                label="Run Criminal Database Scan",
                value=True
            )
            generate_btn = gr.Button("Generate", variant="primary")

        with gr.Column():
            mugshot_out = gr.Image(label="Photorealistic Mugshot")
            sketch_out  = gr.Image(label="Forensic Sketch")
            result_out  = gr.Textbox(label="Database Scan Result", lines=8)

    generate_btn.click(
        fn=forensicai_pipeline,
        inputs=[description, output_style, db_scan],
        outputs=[mugshot_out, sketch_out, result_out]
    )

    gr.Markdown("---")
    gr.Markdown("**ForensicAI** | Arpita Waghmare, Sampada Bandre, Lavanya Tiwade | SIT Nagpur | ICACS 2026")

demo.launch(share=True)
print("✅ Gradio interface launched! Open the public URL above.")

## STEP 10 — Save Outputs to Google Drive (Optional)

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount("/content/drive")

# Create ForensicAI folder in Drive
drive_folder = "/content/drive/MyDrive/ForensicAI_Outputs"
os.makedirs(drive_folder, exist_ok=True)

# Save generated images
mugshot.save(f"{drive_folder}/mugshot_output.png")
sketch.save(f"{drive_folder}/sketch_output.png")

# Save database
if os.path.exists("/content/criminal_database"):
    shutil.copytree(
        "/content/criminal_database",
        f"{drive_folder}/criminal_database",
        dirs_exist_ok=True
    )

print(f"✅ All outputs saved to Google Drive: {drive_folder}")

---
## 📊 Key Results (from paper)

| Metric | Value | Reference |
|--------|-------|----------|
| FID Score — Mugshot | 5.2 | SD v1.5 ref = 3.5 |
| FID Score — Sketch | 6.8 | Target < 10.0 ✅ |
| CLIP Score | 0.31 | SD v1.5 ref = 0.31 ✅ |
| Human Realism (mean) | 4.1 / 5 | Threshold 4.0 ✅ |
| Neg. Prompt Improvement | +40% to +84% | — |
| Face Matching Accuracy | 100% | 5/5 cases |
| Latent vs Pixel Speedup | 48× | 16,384 vs 786,432 values |

## 📐 Core Equations

**Classifier-Free Guidance:**
$$\hat{\epsilon} = \epsilon_u + w \cdot (\epsilon_c - \epsilon_u)$$

**VAE Decoding:**
$$\hat{x} = \text{VAE\_decode}\left(\frac{z_0}{0.18215}\right) \rightarrow 512 \times 512 \times 3$$

**Face Similarity:**
$$S_{ensemble} = \frac{S_{Facenet512} + S_{VGGFace}}{2}, \quad \text{Decision: } S \geq 60\% \rightarrow \text{MATCH}$$

---
*ForensicAI | SIT Nagpur | ICACS 2026*